## 0 — Imports & Setup

In [ ]:
import os, zipfile, random, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import torchvision
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.datasets import ImageFolder

from PIL import Image

print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2 — Data Preprocessing

In [ ]:
DATASET_ZIP  = 'dataset/food11'
DATASET_ROOT = 'dataset/food11_extracted'

if not os.path.isdir(DATASET_ROOT):
    print('Extracting dataset ...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(DATASET_ROOT)
    print('Done.')
else:
    print('Dataset already extracted.')

# Discover split folders (training / validation / evaluation)
for root, dirs, files in os.walk(DATASET_ROOT):
    dirs.sort()
    depth = root.replace(DATASET_ROOT, '').count(os.sep)
    if depth <= 2:
        indent = ' ' * 2 * depth
        print(f'{indent}{os.path.basename(root)}/')

In [ ]:
# Visualize a random subset of training images
vis_dataset = ImageFolder(TRAIN_DIR)  # no transform -> raw PIL images
class_names = vis_dataset.classes

num_images = 9
idxs = random.sample(range(len(vis_dataset)), k=min(num_images, len(vis_dataset)))

cols = 3
rows = int(np.ceil(len(idxs) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, idxs):
    img, label = vis_dataset[idx]
    ax.imshow(img)
    ax.set_title(class_names[label])
    ax.axis("off")

for ax in axes[len(idxs):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Locate the three split directories (handles nested zip structures)
def find_split_dir(root, name_candidates):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() in name_candidates:
                return os.path.join(dirpath, d)
    return None

TRAIN_DIR = find_split_dir(DATASET_ROOT, {'training'})
VAL_DIR   = find_split_dir(DATASET_ROOT, {'validation'})
TEST_DIR  = find_split_dir(DATASET_ROOT, {'evaluation'})

print(f'Train : {TRAIN_DIR}')
print(f'Val   : {VAL_DIR}')
print(f'Test  : {TEST_DIR}')

In [ ]:
# Compute per-channel mean and std on the TRAINING set (resize only, no augmentation)
_loader_transform = T.Compose([
    T.Resize((100, 100)),
    T.ToTensor(),   # [0,1]
])

_stat_dataset = ImageFolder(TRAIN_DIR, transform=_loader_transform)
_stat_loader  = DataLoader(_stat_dataset, batch_size=256, shuffle=False,
                           num_workers=4, pin_memory=False)

mean = torch.zeros(3)
std  = torch.zeros(3)
n_samples = 0

for imgs, _ in _stat_loader:
    # imgs: (B, C, H, W)
    B = imgs.size(0)
    mean += imgs.mean(dim=[0, 2, 3]) * B
    std  += imgs.std(dim=[0, 2, 3])  * B
    n_samples += B

mean /= n_samples
std  /= n_samples

MEAN = mean.tolist()
STD  = std.tolist()
print(f'Channel mean : {[f"{v:.4f}" for v in MEAN]}')
print(f'Channel std  : {[f"{v:.4f}" for v in STD]}')

In [ ]:
# ── Preprocessing pipeline ────────────────────────────────────────────────────
# Training : resize → random horizontal flip (p=0.5)
#            → pad 12 px → random 100×100 crop → normalize
# Val/Test : resize → normalize  (deterministic)

TRAIN_TRANSFORM = T.Compose([
    T.Resize((100, 100)),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(padding=12, fill=0),            # 12-pixel zero-padding on all sides
    T.RandomCrop(100),                    # crop 100×100 from padded 124×124
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

EVAL_TRANSFORM = T.Compose([
    T.Resize((100, 100)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

print('Transforms defined.')

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 4

train_dataset = ImageFolder(TRAIN_DIR, transform=TRAIN_TRANSFORM)
val_dataset   = ImageFolder(VAL_DIR,   transform=EVAL_TRANSFORM)
test_dataset  = ImageFolder(TEST_DIR,  transform=EVAL_TRANSFORM)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
print(f'Classes ({NUM_CLASSES}): {train_dataset.classes}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### Utility Functions (shared across all sections)

In [ ]:
# ── Model factory ─────────────────────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES, pretrained=False):
    """EfficientNet-B0, randomly initialised (no pretrained weights)."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(DEVICE)


# ── One-epoch train / eval ────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


# ── Training loop ─────────────────────────────────────────────────────────────
def run_training(model, train_loader, val_loader, optimizer, criterion,
                 num_epochs, scheduler=None, tag=''):
    """Train for num_epochs, return history dict."""
    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        elapsed = time.time() - t0
        if epoch % 10 == 0 or epoch <= 5 or epoch == num_epochs:
            print(f'[{tag}] Epoch {epoch:3d}/{num_epochs}  '
                  f'loss {tr_loss:.4f}/{vl_loss:.4f}  '
                  f'acc {tr_acc:.3f}/{vl_acc:.3f}  '
                  f'({elapsed:.1f}s)')
    return history


# ── Plotting ──────────────────────────────────────────────────────────────────
def plot_curves(histories, labels, title=''):
    """Plot loss and accuracy curves for one or more experiments."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = plt.cm.tab10.colors

    for i, (hist, lbl) in enumerate(zip(histories, labels)):
        c = colors[i % len(colors)]
        epochs = range(1, len(hist['train_loss']) + 1)
        axes[0].plot(epochs, hist['train_loss'], color=c, linestyle='-',  label=f'{lbl} train')
        axes[0].plot(epochs, hist['val_loss'],   color=c, linestyle='--', label=f'{lbl} val')
        axes[1].plot(epochs, hist['train_acc'],  color=c, linestyle='-',  label=f'{lbl} train')
        axes[1].plot(epochs, hist['val_acc'],    color=c, linestyle='--', label=f'{lbl} val')

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} — Loss'); axes[0].legend()
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{title} — Accuracy'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_")}.png', dpi=150)
    plt.show()


def report_final(history, tag):
    print(f'\n=== {tag} — Final Results ===')
    print(f'  Train loss : {history["train_loss"][-1]:.4f}')
    print(f'  Val   loss : {history["val_loss"][-1]:.4f}')
    print(f'  Train acc  : {history["train_acc"][-1]:.4f}')
    print(f'  Val   acc  : {history["val_acc"][-1]:.4f}')


print('Utility functions ready.')

---
## 3 — Learning Rate
Three runs: LR ∈ {0.1, 0.025, 0.001}  |  SGD momentum=0.9  |  No weight decay  |  No schedule  |  15 epochs

In [ ]:
LR_CANDIDATES = [0.1, 0.025, 0.001]
NUM_EPOCHS_LR = 15

lr_histories = {}

for lr in LR_CANDIDATES:
    print(f'\n──── LR = {lr} ────────────────────────────────────')
    model     = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                weight_decay=0.0)
    hist = run_training(model, train_loader, val_loader,
                        optimizer, criterion,
                        num_epochs=NUM_EPOCHS_LR, tag=f'LR={lr}')
    lr_histories[lr] = hist
    report_final(hist, f'LR={lr}')

print('\nSection 3 done.')

In [ ]:
plot_curves(
    histories=[lr_histories[lr] for lr in LR_CANDIDATES],
    labels   =[f'LR={lr}' for lr in LR_CANDIDATES],
    title='Section 3 Learning Rate'
)

# Identify best LR by final training loss
BEST_LR = min(LR_CANDIDATES, key=lambda lr: lr_histories[lr]['train_loss'][-1])
print(f'Best LR (lowest training loss): {BEST_LR}')

---
## 4 — Learning Rate Schedule
### 4.1 Cosine Annealing — Description
Cosine annealing smoothly reduces the learning rate following a cosine curve:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_0 - \eta_{\min})\left(1 + \cos\!\left(\frac{\pi\, t}{T}\right)\right)$$

where $\eta_0$ is the initial learning rate, $\eta_{\min}$ is the minimum (here 0), $t$ is the current epoch and $T$ is the total number of epochs.  
Intuitively the schedule starts fast, then decelerates gently, allowing the optimiser to refine the solution in flat basins rather than oscillating around a minimum.

### 4.2 Experiment
Two conditions, both 300 epochs with the best initial LR from Section 3:
1. **Constant LR** — no schedule
2. **Cosine annealing** — LR decays from `BEST_LR` to 0 over 300 epochs

In [ ]:
NUM_EPOCHS_SCHED = 300

# ── Condition 1: constant LR ──────────────────────────────────────────────────
print('── Constant LR ──')
model_const = build_model()
opt_const   = torch.optim.SGD(model_const.parameters(), lr=BEST_LR,
                               momentum=0.9, weight_decay=0.0)
criterion   = nn.CrossEntropyLoss()
hist_const  = run_training(model_const, train_loader, val_loader,
                            opt_const, criterion,
                            num_epochs=NUM_EPOCHS_SCHED, tag='Constant')
report_final(hist_const, 'Constant LR')

# ── Condition 2: cosine annealing ─────────────────────────────────────────────
print('\n── Cosine Annealing ──')
model_cos = build_model()
opt_cos   = torch.optim.SGD(model_cos.parameters(), lr=BEST_LR,
                             momentum=0.9, weight_decay=0.0)
sched_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_cos, T_max=NUM_EPOCHS_SCHED, eta_min=0.0)
hist_cos  = run_training(model_cos, train_loader, val_loader,
                          opt_cos, criterion,
                          num_epochs=NUM_EPOCHS_SCHED,
                          scheduler=sched_cos, tag='CosineAnneal')
report_final(hist_cos, 'Cosine Annealing')

print('\nSection 4 done.')

In [ ]:
plot_curves(
    histories=[hist_const, hist_cos],
    labels   =['Constant LR', 'Cosine Annealing'],
    title='Section 4 LR Schedule'
)

---
## 5 — Weight Decay
Best LR + cosine annealing + weight decay λ ∈ {5×10⁻⁴, 1×10⁻⁴} | 300 epochs

In [ ]:
WD_VALUES = [5e-4, 1e-4]
NUM_EPOCHS_WD = 300

wd_histories = {}

for wd in WD_VALUES:
    print(f'\n──── Weight Decay λ = {wd} ────────────────────────')
    model_wd = build_model()
    opt_wd   = torch.optim.SGD(model_wd.parameters(), lr=BEST_LR,
                                momentum=0.9, weight_decay=wd)
    sched_wd = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_wd, T_max=NUM_EPOCHS_WD, eta_min=0.0)
    hist_wd  = run_training(model_wd, train_loader, val_loader,
                             opt_wd, criterion,
                             num_epochs=NUM_EPOCHS_WD,
                             scheduler=sched_wd, tag=f'WD={wd}')
    wd_histories[wd] = hist_wd
    report_final(hist_wd, f'Weight Decay λ={wd}')

print('\nSection 5 done.')

In [ ]:
# Compare: no weight decay (cosine, Section 4) vs two WD values
plot_curves(
    histories=[hist_cos] + [wd_histories[wd] for wd in WD_VALUES],
    labels   =['No WD (cosine)'] + [f'WD λ={wd}' for wd in WD_VALUES],
    title='Section 5 Weight Decay'
)

---
## 6 — Data Augmentation: Mixup (α = 0.2)

### Beta Distribution PDF (α = β = 0.2)

In [ ]:
from scipy.stats import beta as beta_dist

alpha_mixup = 0.2
x = np.linspace(1e-4, 1 - 1e-4, 500)
pdf = beta_dist.pdf(x, a=alpha_mixup, b=alpha_mixup)

plt.figure(figsize=(6, 3.5))
plt.plot(x, pdf, color='steelblue', linewidth=2)
plt.fill_between(x, pdf, alpha=0.25, color='steelblue')
plt.xlabel('λ (mixing coefficient)')
plt.ylabel('Density')
plt.title(f'Beta(α={alpha_mixup}, β={alpha_mixup}) — Mixup mixing coefficient PDF')
plt.tight_layout()
plt.savefig('beta_pdf.png', dpi=150)
plt.show()
print('The U-shaped PDF concentrates mass near 0 and 1, '
      'so most mixed samples are close to one of the two originals.')

In [ ]:
# ── Mixup helpers ─────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    idx = torch.randperm(batch_size, device=x.device)
    mixed_x  = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch_mixup(model, loader, optimizer, criterion, alpha=0.2):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        mixed, y_a, y_b, lam = mixup_data(imgs, labels, alpha=alpha)
        optimizer.zero_grad()
        logits = model(mixed)
        loss   = mixup_criterion(criterion, logits, y_a, y_b, lam)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        # Accuracy: prediction vs the dominant label
        pred = logits.argmax(1)
        correct += (lam * (pred == y_a).float() +
                    (1 - lam) * (pred == y_b).float()).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


def run_training_mixup(model, train_loader, val_loader, optimizer, criterion,
                       num_epochs, scheduler=None, alpha=0.2, tag=''):
    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch_mixup(model, train_loader,
                                                 optimizer, criterion, alpha)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        elapsed = time.time() - t0
        if epoch % 10 == 0 or epoch <= 5 or epoch == num_epochs:
            print(f'[{tag}] Epoch {epoch:3d}/{num_epochs}  '
                  f'loss {tr_loss:.4f}/{vl_loss:.4f}  '
                  f'acc {tr_acc:.3f}/{vl_acc:.3f}  '
                  f'({elapsed:.1f}s)')
    return history


print('Mixup helpers ready.')

In [ ]:
# Best setup so far: BEST_LR + cosine annealing + best weight decay
BEST_WD = min(WD_VALUES, key=lambda wd: wd_histories[wd]['val_loss'][-1])
print(f'Best weight decay from Section 5: {BEST_WD}')

In [ ]:
NUM_EPOCHS_MIX = 300
ALPHA_MIXUP    = 0.2

print('── No Mixup (baseline for comparison) ──')
model_base = build_model()
opt_base   = torch.optim.SGD(model_base.parameters(), lr=BEST_LR,
                              momentum=0.9, weight_decay=BEST_WD)
sched_base = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_base, T_max=NUM_EPOCHS_MIX, eta_min=0.0)
hist_base  = run_training(model_base, train_loader, val_loader,
                           opt_base, criterion,
                           num_epochs=NUM_EPOCHS_MIX,
                           scheduler=sched_base, tag='NoMixup')
report_final(hist_base, 'No Mixup')

print('\n── With Mixup (α=0.2) ──')
model_mix = build_model()
opt_mix   = torch.optim.SGD(model_mix.parameters(), lr=BEST_LR,
                             momentum=0.9, weight_decay=BEST_WD)
sched_mix = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_mix, T_max=NUM_EPOCHS_MIX, eta_min=0.0)
hist_mix  = run_training_mixup(model_mix, train_loader, val_loader,
                                opt_mix, criterion,
                                num_epochs=NUM_EPOCHS_MIX,
                                scheduler=sched_mix,
                                alpha=ALPHA_MIXUP, tag='Mixup')
report_final(hist_mix, 'Mixup α=0.2')

print('\nSection 6 done.')

In [ ]:
plot_curves(
    histories=[hist_base, hist_mix],
    labels   =['No Mixup', 'Mixup α=0.2'],
    title='Section 6 Mixup Augmentation'
)

In [ ]:
# ── Final test-set accuracy (reported only at the end) ────────────────────────
criterion_eval = nn.CrossEntropyLoss()

test_loss_base, test_acc_base = evaluate(model_base, test_loader, criterion_eval)
test_loss_mix,  test_acc_mix  = evaluate(model_mix,  test_loader, criterion_eval)

print('=== Hold-out Test Set Results ===')
print(f'  No Mixup  — loss: {test_loss_base:.4f}  acc: {test_acc_base:.4f} ({test_acc_base*100:.2f}%)')
print(f'  Mixup 0.2 — loss: {test_loss_mix:.4f}  acc: {test_acc_mix:.4f} ({test_acc_mix*100:.2f}%)')